In [ ]:
import sys
import os
import hashlib
import statistics
from pyspark import SparkContext, SparkConf
from pyspark.streaming import StreamingContext

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

conf = SparkConf().setAppName("FM_Median_Robust").setMaster("local[2]")

try:
    sc.stop()
except:
    pass

sc = SparkContext(conf=conf)
sc.setLogLevel("WARN")

ssc = StreamingContext(sc, 5)

input_directory = "file:///home/reza/HW4/stream_input"
dstream = ssc.textFileStream(input_directory)

HASH_FUNCS = ['md5', 'sha1', 'sha224', 'sha256']

def get_trailing_zeros(binary_str):
    if not binary_str: return 0
    s = str(binary_str)
    return len(s) - len(s.rstrip('0'))

def apply_fm_hashing(plate):
    zeros_list = []
    plate_encoded = plate.strip().encode('utf-8')
    for alg in HASH_FUNCS:
        h_obj = getattr(hashlib, alg)(plate_encoded)
        h_hex = h_obj.hexdigest()
        h_bin = bin(int(h_hex, 16))[2:]
        zeros_list.append(get_trailing_zeros(h_bin))
    return zeros_list

def parse_camera_pass(line):
    try:
        parts = line.split(',')
        if len(parts) < 3: return None
        camera_id = parts[1].strip()
        plate = parts[2].strip()
        return (camera_id, plate)
    except:
        return None

parsed_dstream = dstream.map(parse_camera_pass).filter(lambda x: x is not None)

WINDOW_DURATION = 30
SLIDE_DURATION = 10
windowed_dstream = parsed_dstream.window(WINDOW_DURATION, SLIDE_DURATION)

def process_fm(time, rdd):
    try:
        if not rdd.isEmpty():
            rdd.cache()
            
            actual_global = rdd.map(lambda x: x[1]).distinct().count()
            actual_per_cam = rdd.map(lambda x: (x[0], x[1])).distinct() \
                                .map(lambda x: (x[0], 1)).reduceByKey(lambda a,b: a+b).collectAsMap()
            
            fm_rdd = rdd.map(lambda x: (x[0], apply_fm_hashing(x[1])))
            
            def elem_max(l1, l2):
                return [max(a, b) for a, b in zip(l1, l2)]

            global_zeros = fm_rdd.map(lambda x: x[1]).reduce(elem_max)
            
            
            median_zeros_global = statistics.median(global_zeros)
            est_global = int(2 ** median_zeros_global)
            
            err_global = 0.0
            if actual_global > 0:
                err_global = abs(actual_global - est_global) / actual_global * 100

            print(f"========= Time: {time} =========")
            print(f"Global Actual: {actual_global} | Estimated: {est_global} | Error: {err_global:.2f}%")
            print("-" * 60)
            
            
            cam_estimates = fm_rdd.reduceByKey(elem_max) \
                                  .mapValues(lambda z: int(2 ** statistics.median(z))) \
                                  .collect()
            
            cam_estimates.sort(key=lambda x: x[1], reverse=True)
            
            print("Camera Estimates (Top 10):")
            for cam, est in cam_estimates[:10]:
                act = actual_per_cam.get(cam, 0)
                err = 0.0
                if act > 0: err = abs(act - est) / act * 100
                print(f"Cam {cam:<5} | Act: {act:<5} | Est: {est:<5} | Err: {err:.2f}%")
            print("\n")
            
            rdd.unpersist()
    except Exception as e:
        print(e)
        pass

windowed_dstream.foreachRDD(process_fm)

ssc.start()